Défi quotidien : Pipelines LangChain avec LLM open-source

LangChain Pipelines avec des LLMs Open-Source


Partie 1 : Configuration de l'environnement

In [ ]:
# Vérifier le matériel (optionnel)
!nvidia-smi || echo "CPU runtime"

# Installer les dépendances avec les versions spécifiées
!pip install "transformers==4.37.2" "langchain==0.1.7" "langchain-community==0.0.20" "langchain-core==0.1.23"

/bin/bash: line 1: nvidia-smi: command not found
CPU runtime


Partie 2 : Chargez un petit modèle ouvert et construisez votre première chaîne LLM

Objectif : Réécrire un texte dans un style plus simple.
python
Copier







In [ ]:
# Importer les bibliothèques nécessaires
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain import PromptTemplate, LLMChain

# Choisir un petit modèle pour le CPU
model_name = "google/flan-t5-small"

# Charger le tokenizer et le modèle
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Créer un pipeline de génération
gen_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen_pipeline)

# Construire le prompt et la LLMChain pour la réécriture
template = "Rewrite this text to be simpler for beginners: {text}"
prompt = PromptTemplate(template=template, input_variables=["text"])
chain = LLMChain(prompt=prompt, llm=llm)

# Tester avec un exemple
sample_text = "LangChain helps you build LLM apps by composing prompts, models, and tools."
rewritten = chain.run(text=sample_text)
print("🔄 Texte réécrit :\n", rewritten)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


🔄 Texte réécrit :
 LangChain helps you build LLM apps by composing prompts, models, and tools.


Sortie attendue :

In [ ]:
# 🔄 Texte réécrit :
# LangChain helps you create apps with large language models by combining prompts, models, and tools.

Partie 3 : Pipeline en 2 étapes avec RunnableSequence


Objectif : Résumer un paragraphe → Extraire 3 points clés.
python
Copier







In [ ]:
from langchain import PromptTemplate

# Définir les templates pour chaque étape
summary_template = "Summarize this paragraph in one sentence: {paragraph}"
summary_prompt = PromptTemplate(template=summary_template, input_variables=["paragraph"])

bullets_template = "Turn this summary into exactly 3 bullet points: {summary}"
bullets_prompt = PromptTemplate(template=bullets_template, input_variables=["summary"])

# Première étape : paragraphe → résumé
summary_chain = summary_prompt | llm

# Pipeline complet : résumé → points clés
summarize_then_bullets = (
    {"summary": summary_chain}  # Crée un dictionnaire avec la sortie du résumé
    | bullets_prompt
    | llm
)

# Tester avec un paragraphe
paragraph = """LangChain is a framework for building applications with large language models by composing prompts, models, and tools. It supports chains, agents, and retrieval workflows."""
bullets_output = summarize_then_bullets.invoke({"paragraph": paragraph})
print("📌 Résumé + Points clés :\n", bullets_output)

📌 Résumé + Points clés :
 Use LangChain to build applications with large language models.


Sortie attendue :

In [ ]:
# 📌 Résumé + Points clés :
# - LangChain is a framework for building LLM applications.
# - It composes prompts, models, and tools.
# - It supports chains, agents, and retrieval workflows.

Partie 4 (Bonus) : Chaîne de conversation avec mémoire


Objectif : Illustrer la mémoire avec ConversationChain.


In [ ]:
# Importer les modules nécessaires
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

# Configurer la mémoire
memory = ConversationBufferMemory()
convo = ConversationChain(llm=llm, memory=memory, verbose=False)

# Tester avec deux tours de dialogue
reply1 = convo.predict(input="Hi there! What's LangChain?")
reply2 = convo.predict(input="Can it help me build a simple chatbot?")

print("💬 Tour 1 :", reply1)
print("💬 Tour 2 :", reply2)

💬 Tour 1 : LangChain is a fictional character in the "LangChain" series.
💬 Tour 2 : It's a simple chatbot.


Sortie attendue :

In [ ]:
# 💬 Tour 1: LangChain is a framework for building applications with large language models.
# 💬 Tour 2: Yes, it can help you build a simple chatbot by composing prompts, models, and tools.